# Lab Assignment 1: Contextual Bandits

**Oliver Grudzinski**

Self-contained. Every class and function used is defined in this notebook.

Section 6.1 runs the full experiment schedule, which takes about 80 minutes.
It caches to `results/*.csv` and reloads from there on later runs, so it only
costs that once.

## 1. Setup

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn joblib

In [ ]:
import csv
import itertools
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

np.random.seed(0)

# ---- environment constants ----
N_ACTIONS = 5
P_CLICK_CORRECT = 0.9    # click probability when the agent picks the right article
P_CLICK_WRONG = 0.05     # click probability otherwise

# Expected reward of a uniformly random policy:
#   (1/5)(0.9) + (4/5)(0.05) = 0.22
RANDOM_POLICY_EXPECTED_REWARD = (
    (1 / N_ACTIONS) * P_CLICK_CORRECT + (1 - 1 / N_ACTIONS) * P_CLICK_WRONG
)

# ---- experiment grid ----
N_TURNS = 5000
T_VALUES = [100, 250, 500]
B_VALUES = [100, 1000]
EPS_VALUES = [0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

# Fixed point for the Section 6 comparisons, so the only thing changing
# between the baseline and a broken environment is the environment itself.
BROKEN_T = 500
BROKEN_B = 100
BROKEN_EPS = [0.0, 0.05, 0.1, 0.2, 0.3]

DRIFT_TURN = 2500      # turn at which NonStationaryEnv permutes the labels
ROLLING_WINDOW = 500   # trailing window for the rolling reward curve
CURVE_STRIDE = 25      # downsampling stride for saved curves

RESULTS_DIR = "results"
FIG_DIR = "figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("random policy expected reward:", RANDOM_POLICY_EXPECTED_REWARD)

## 2. The environment

In [ ]:
class Contextual_Bandit_generator:
    """Simulator for the article recommendation contextual bandit.

    The simulator holds a fixed population of synthetic users. Each user is a
    5-dimensional feature vector X paired with a hidden label `best_article`
    giving the one article out of five that this kind of user actually wants.
    On every turn the agent is shown one user drawn at random from that
    population, picks an article, and gets back a click or no click.

    Two components:
      step()     draws the next user and exposes its context.
      reward(a)  scores the article the agent picked for that user.

    Parameters
    ----------
    config : dict
        Must contain 'n_samples', the size of the user population to generate.
    seed : int, optional
        If given, the simulator draws from its own RandomState instead of the
        global numpy random state. Behavior is identical to the original class
        when seed is None. This exists only so parallel runs are reproducible.

    Attributes
    ----------
    population : list of [X, y]
        population[0] is the (n_samples, 5) engineered feature matrix.
        population[1] is the (n_samples,) array of hidden best-article labels.
    state : dict
        Full interaction log: every context shown, action taken, reward given.
    current_context : dict
        The user currently on screen: 'X' and the hidden 'best_article'.
    """

    name = "baseline"

    def __init__(self, config, seed=None):
        # Interaction log. Grows by one row per turn.
        self.state = {'X': [], 'article': [], 'reward': []}

        # Using a RandomState when seeded keeps runs reproducible across worker
        # processes. np.random (the module) exposes the same .choice/.binomial
        # API, so the unseeded path behaves exactly like the original class.
        self.rng = np.random if seed is None else np.random.RandomState(seed)

        self.population = [[], []]

        # Build the user population once, up front. This is what makes the
        # environment stationary: the same fixed pool is sampled every turn.
        #
        #   n_features=5, n_informative=5, n_redundant=0, n_repeated=0
        #       all five features carry signal, none are copies or linear
        #       combinations of each other
        #   n_classes=5
        #       five articles, so five possible "best" labels
        #   flip_y=0.2
        #       20 percent of the labels are randomly reassigned after the
        #       clusters are built. This is label noise, and it is the reason
        #       no model of X can perfectly recover best_article.
        #   shift / scale
        #       per-feature affine transform applied to the raw gaussians. The
        #       shift of 10 and scale of 7 on feature 0, and shift 4 on
        #       feature 1, are what push those columns into ranges that look
        #       like real user attributes after feature_engineer runs.
        #   random_state=8
        #       fixed, so the population is the same every time.
        self.population[0], self.population[1] = make_classification(
            n_samples=config['n_samples'],
            n_features=5,
            n_informative=5,
            n_classes=5,
            n_redundant=0,
            n_repeated=0,
            flip_y=0.2,
            shift=[10, 4, 0, 0, 0],
            scale=[7, 1, 1, 1, 1],
            random_state=8
        )

        self.population[0] = self.feature_engineer(self.population[0])
        self.current_context = {'X': [], 'best_article': []}

    def feature_engineer(self, Data):
        """Turn raw gaussian columns into a plausible user profile.

        The generator produces continuous gaussians. This method discretizes
        them so the context looks like something a real recommender would log:

          col 0 -> binary flag  (1 if positive, else 0)
          col 1 -> binary flag  (1 if positive, else 0)
          col 2 -> left continuous, the only real-valued feature
          col 3 -> clipped to [0, 5] then rounded, so a small integer count
          col 4 -> rounded to an integer, and because of the shift/scale above
                   it lands in a range that reads like an age

        Note this is a lossy transform. Binarizing columns 0 and 1 throws away
        information that make_classification put there, which lowers the
        accuracy ceiling further on top of flip_y.
        """
        Data[:, 0] = (Data[:, 0] > 0) * 1
        Data[:, 1] = (Data[:, 1] > 0) * 1
        Data[:, 3][(Data[:, 3] < 0)] = 0
        Data[:, 3][(Data[:, 3] > 5)] = 5
        Data[:, 3] = np.round(Data[:, 3])
        Data[:, 4] = np.round(Data[:, 4])
        return Data

    def _draw(self):
        """Sample one user uniformly at random from the population.

        Returns (x, best_article). Split out from step() so the broken
        subclasses can corrupt the draw without reimplementing the logging.
        """
        sample = self.rng.choice(len(self.population[1]))
        return self.population[0][sample], self.population[1][sample]

    def step(self):
        """Advance one turn: draw a new user and put them on screen.

        Sampling is uniform with replacement, so contexts are i.i.d. and
        nothing about the environment changes over time. The drawn context is
        appended to the running log and stored in current_context, where
        reward() will find it.

        The hidden best_article is stored but never handed to the agent. It is
        available here only so experiments can measure how often the agent
        picked the truly correct article.
        """
        x, ba = self._draw()

        if len(self.state['X']) == 0:
            self.state['X'] = x
        else:
            self.state['X'] = np.vstack((self.state['X'], x))

        self.current_context['X'] = x
        self.current_context['best_article'] = ba

    def reward(self, a):
        """Score action `a` against the user currently on screen.

        The click is a coin flip whose bias depends only on whether the agent
        picked the hidden best article:

            a == best_article  ->  click with probability 0.9
            a != best_article  ->  click with probability 0.05

        So the reward is binary and stochastic even when the agent is right.
        The action and the realized reward are appended to the log.

        Returns the realized reward.
        """
        if self.current_context['best_article'] == a:
            response = self.rng.binomial(n=1, p=P_CLICK_CORRECT, size=1)[0]
        else:
            response = self.rng.binomial(n=1, p=P_CLICK_WRONG, size=1)[0]

        if len(self.state['X']) == 0:
            self.state['article'] = np.array(a)
            self.state['reward'] = np.array(response)
        else:
            self.state['article'] = np.append(self.state['article'], a)
            self.state['reward'] = np.append(self.state['reward'], response)

        return response

### 2.1 Intended setup and use

In [ ]:
env_config = {'n_samples': 100000}
env = Contextual_Bandit_generator(env_config, seed=0)

env.step()
print("context X          :", env.current_context['X'])
print("hidden best_article:", env.current_context['best_article'])

a = 2
r = env.reward(a)
print("action taken       :", a)
print("reward             :", r)

In [ ]:
demo = Contextual_Bandit_generator({'n_samples': 100000}, seed=1)
peek = pd.DataFrame(demo.population[0][:8],
                    columns=['feature 1', 'feature 2', 'feature 3',
                             'feature 4', 'feature 5'])
peek.insert(0, 't', np.arange(1, 9))
peek['best_article (hidden)'] = demo.population[1][:8]
display(peek)

print("population size      :", demo.population[0].shape)
print("article distribution :",
      np.bincount(demo.population[1]) / len(demo.population[1]))

## 3. Random actions for 10, 100, 1000 and 10000 turns

In [ ]:
def run_random_baseline(env, n_turns, seed=None):
    """Take uniformly random actions for n_turns.

    This is the reference point everything else is measured against. In theory
    the average reward is (1/5)(0.9) + (4/5)(0.05) = 0.22 regardless of how
    many turns are run; what changes with n_turns is only the variance.
    """
    rng = np.random.RandomState(seed)
    rewards = np.zeros(n_turns, dtype=float)
    matches = np.zeros(n_turns, dtype=bool)

    for i in range(n_turns):
        env.step()
        a = int(rng.randint(N_ACTIONS))
        rewards[i] = env.reward(a)
        matches[i] = int(env.current_context['best_article']) == a

    return {'reward': rewards,
            'mean_reward': float(rewards.mean()),
            'match_rate': float(matches.mean())}

In [ ]:
rows = []
for n_turns in [10, 100, 1000, 10000]:
    for seed in range(30):
        e = Contextual_Bandit_generator({'n_samples': 100000}, seed=seed)
        out = run_random_baseline(e, n_turns, seed=seed)
        rows.append({'n_turns': n_turns, 'seed': seed,
                     'mean_reward': out['mean_reward'],
                     'match_rate': out['match_rate']})

rand_df = pd.DataFrame(rows)
rand_df.to_csv(os.path.join(RESULTS_DIR, 'random_baseline.csv'), index=False)

rand_summary = rand_df.groupby('n_turns')['mean_reward'].agg(
    ['mean', 'std', 'min', 'max'])
rand_summary['theory'] = RANDOM_POLICY_EXPECTED_REWARD
display(rand_summary.round(4))

In [ ]:
def plot_random_baseline(df):
    """Bar chart of average reward under uniform random actions."""
    g = df.groupby('n_turns')['mean_reward']
    turns = sorted(df['n_turns'].unique())
    means = [g.get_group(t).mean() for t in turns]
    stds = [g.get_group(t).std(ddof=1) for t in turns]
    n_reps = len(df[df['n_turns'] == turns[0]])

    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(turns))
    ax.bar(x, means, yerr=stds, capsize=6, color='#4c72b0',
           edgecolor='black', linewidth=0.6, alpha=0.9)
    ax.axhline(RANDOM_POLICY_EXPECTED_REWARD, color='crimson', linestyle='--',
               linewidth=1.8,
               label='theory: 0.2(0.9) + 0.8(0.05) = %.2f'
                     % RANDOM_POLICY_EXPECTED_REWARD)

    for xi, m, s in zip(x, means, stds):
        ax.text(xi, m + s + 0.012, '%.3f\n(sd %.3f)' % (m, s),
                ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(['%d turns' % t for t in turns])
    ax.set_ylabel('average reward (click rate)')
    ax.set_title('Random action baseline, %d repetitions per bar' % n_reps)
    ax.set_ylim(0, max(np.array(means) + np.array(stds)) * 1.35)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    fig.savefig(os.path.join(FIG_DIR, 'q5_random_baseline_bar.png'),
                dpi=150, bbox_inches='tight')
    plt.show()


plot_random_baseline(rand_df)

## 4. The contextual bandit agent

In [ ]:
class _ConstantPredictor:
    """Fallback for an arm with too little data to fit a classifier.

    An arm can end up with zero observations, or with observations that are
    all the same class, in which case a RandomForestClassifier either refuses
    to fit or returns a single-column predict_proba. Rather than special-case
    that everywhere, such arms get a constant predictor returning the observed
    click rate (or a shared prior when the arm was never played).
    """

    def __init__(self, p):
        self.p = float(p)

    def predict_pclick(self, X):
        return np.full(len(X), self.p)


class _ForestPredictor:
    """Thin wrapper so forests and constants share one interface."""

    def __init__(self, model):
        self.model = model

    def predict_pclick(self, X):
        return self.model.predict_proba(X)[:, 1]


class ContextualBanditAgent:
    """Five per-article function approximators plus an epsilon-greedy policy.

    One binary classifier f_a per article, each estimating P(click | X, a).
    Each f_a is trained only on the turns where article a was actually shown,
    with the observed click as the label. This per-arm split is the whole
    point: it is what lets the agent reason about actions it did not take.

    The classifiers are untuned RandomForestClassifier instances, left at
    sklearn defaults (100 trees, unlimited depth) as the assignment requires.

    Parameters
    ----------
    n_actions : int
    seed : int, optional
        Seeds both the forests and the exploration/tie-breaking randomness.
    rf_n_jobs : int
        Threads per forest. Keep at 1 when running many configurations in
        parallel processes, otherwise the workers oversubscribe the CPU.
    """

    def __init__(self, n_actions=N_ACTIONS, seed=None, rf_n_jobs=1):
        self.n_actions = n_actions
        self.seed = seed
        self.rf_n_jobs = rf_n_jobs
        self.rng = np.random.RandomState(seed)
        # Until the first fit, every arm looks equally good.
        self.models = [_ConstantPredictor(0.0) for _ in range(n_actions)]
        self.is_fitted = False

    def fit(self, X, actions, rewards):
        """Refit all five function approximators on the history so far.

        For each arm the rows where that arm was played are sliced out and a
        fresh forest is fit on them. Arms with no data, or with only one
        observed class, fall back to a constant predictor.
        """
        X = np.asarray(X, dtype=float)
        actions = np.asarray(actions)
        rewards = np.asarray(rewards)

        prior = float(rewards.mean()) if len(rewards) else 0.0

        for a in range(self.n_actions):
            mask = actions == a
            n_a = int(mask.sum())

            if n_a == 0:
                # Never played. Give it the global prior so it is neither
                # ruled out nor artificially favored.
                self.models[a] = _ConstantPredictor(prior)
                continue

            y_a = rewards[mask]

            if len(np.unique(y_a)) < 2:
                # All clicks or all non-clicks. A classifier has nothing to
                # separate, so just report the observed rate.
                self.models[a] = _ConstantPredictor(y_a.mean())
                continue

            model = RandomForestClassifier(
                random_state=None if self.seed is None else self.seed + a,
                n_jobs=self.rf_n_jobs,
            )
            model.fit(X[mask], y_a)
            self.models[a] = _ForestPredictor(model)

        self.is_fitted = True

    def predict_scores(self, x):
        """Return the five estimates f_a(x) = P(click | x, a)."""
        x2d = np.asarray(x, dtype=float).reshape(1, -1)
        return np.array([self.models[a].predict_pclick(x2d)[0]
                         for a in range(self.n_actions)])

    def act(self, x, epsilon):
        """Epsilon-greedy action selection.

        With probability epsilon pick uniformly at random, otherwise take the
        argmax of the five scores. Ties are broken at random rather than by
        taking the lowest index, which matters early on when several arms
        still share the same constant fallback value.

        Returns (action, was_greedy).
        """
        if self.rng.random_sample() < epsilon:
            return int(self.rng.randint(self.n_actions)), False
        scores = self.predict_scores(x)
        best = np.flatnonzero(scores == scores.max())
        return int(self.rng.choice(best)), True

### 4.1 Algorithm 1

In [ ]:
def run_bandit(env, T, b, epsilon, n_turns=N_TURNS, seed=None,
               window=None, rf_n_jobs=1):
    """Run Algorithm 1 from the assignment.

    For t = 1..n_turns:
        draw a context with env.step()
        if t <= T:            pick an action uniformly at random
        else:                 pick with epsilon-greedy over f_a(X)
        collect the reward with env.reward(a)
        every b turns (once t > T) refit all five f_a

    Design decision: the models are fit once at t = T, the end of the pure
    exploration phase, so the very first greedy decision at t = T+1 has
    something to act on. After that they are refit every b turns. Without that
    initial fit the agent would spend its first b greedy turns choosing
    arbitrarily.

    Parameters
    ----------
    env : an environment instance
    T : int
        Length of the pure exploration (explore-then-commit) phase.
    b : int
        Refit the function approximators every b turns.
    epsilon : float
        Exploration rate after turn T.
    n_turns : int
    seed : int, optional
    window : int, optional
        If set, refit on only the most recent `window` observations instead of
        the whole history. This is the sliding-window fix for the
        non-stationary environment; leave it None for plain Algorithm 1.
    rf_n_jobs : int

    Returns
    -------
    dict with per-turn arrays and summary scalars.
    """
    t_start = time.perf_counter()

    agent = ContextualBanditAgent(
        seed=None if seed is None else seed + 10000, rf_n_jobs=rf_n_jobs)
    explore_rng = np.random.RandomState(None if seed is None else seed + 20000)

    # The agent's own log. Kept separately from env.state because env.state
    # uses vstack, which is quadratic, and because the agent should only ever
    # see what it actually observed.
    X_log = None
    a_log = np.zeros(n_turns, dtype=int)
    r_log = np.zeros(n_turns, dtype=float)
    greedy_log = np.zeros(n_turns, dtype=bool)
    match_log = np.zeros(n_turns, dtype=bool)

    n_refits = 0

    for i in range(n_turns):
        t = i + 1

        env.step()
        x = np.asarray(env.current_context['X'], dtype=float)

        if X_log is None:
            X_log = np.zeros((n_turns, x.shape[0]))
        X_log[i] = x

        if t <= T:
            a = int(explore_rng.randint(N_ACTIONS))
            greedy = False
        else:
            a, greedy = agent.act(x, epsilon)

        r = env.reward(a)

        a_log[i] = a
        r_log[i] = r
        greedy_log[i] = greedy
        match_log[i] = int(env.current_context['best_article']) == a

        # Fit at the end of exploration, then every b turns after that.
        if t == T or (t > T and (t - T) % b == 0):
            lo = 0 if window is None else max(0, (i + 1) - window)
            agent.fit(X_log[lo:i + 1], a_log[lo:i + 1], r_log[lo:i + 1])
            n_refits += 1

    elapsed = time.perf_counter() - t_start
    tail = min(1000, n_turns)

    return {
        'reward': r_log,
        'action': a_log,
        'greedy': greedy_log,
        'match_best': match_log,
        'mean_reward': float(r_log.mean()),
        'mean_reward_last': float(r_log[-tail:].mean()),
        'mean_reward_post_T': float(r_log[T:].mean()) if T < n_turns else np.nan,
        'match_rate': float(match_log.mean()),
        'match_rate_last': float(match_log[-tail:].mean()),
        'n_refits': n_refits,
        'seconds': elapsed,
    }

### 4.2 One run

In [ ]:
demo_env = Contextual_Bandit_generator({'n_samples': 100000}, seed=7)
demo_out = run_bandit(demo_env, T=500, b=100, epsilon=0.05,
                      n_turns=5000, seed=7)

print("mean reward, all 5000 turns : %.3f" % demo_out['mean_reward'])
print("mean reward, last 1000 turns: %.3f" % demo_out['mean_reward_last'])
print("picked the best article     : %.1f%% of the last 1000 turns"
      % (100 * demo_out['match_rate_last']))
print("refits performed            : %d" % demo_out['n_refits'])

In [ ]:
r = demo_out['reward']
running = np.cumsum(r) / np.arange(1, len(r) + 1)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(running, linewidth=1.6, label='CB agent (T=500, b=100, eps=0.05)')
ax.axhline(RANDOM_POLICY_EXPECTED_REWARD, color='crimson', linestyle='--',
           label='random policy (0.22)')
ax.axvline(500, color='grey', linestyle=':', label='end of exploration (T)')
ax.set_xlabel('turn'); ax.set_ylabel('running average reward')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. Breaking the environment

In [ ]:
class NonStationaryEnv(Contextual_Bandit_generator):
    """Break it: article preferences change part way through the run.

    At turn `drift_turn` a fixed permutation is applied to the hidden labels,
    so the audience that used to want article 0 now wants article 3, and so
    on. Everything the agent learned before the drift becomes actively
    misleading.

    Why this is realistic: article inventory rotates, editors swap headlines,
    a seasonal event changes what people want. The user population is the same
    but the right answer for them is not.

    Assumption violated: stationarity. Algorithm 1 assumes the reward
    distribution for a given (context, action) pair is fixed, which lets it
    pool all history into one fit. Here old data and new data disagree.
    """

    name = "nonstationary"

    def __init__(self, config, seed=None, drift_turn=DRIFT_TURN,
                 permutation=None):
        super().__init__(config, seed=seed)
        self.t = 0
        self.drift_turn = drift_turn
        # Derangement: no article maps to itself, so the drift is total.
        self.permutation = (np.array(permutation) if permutation is not None
                            else np.array([3, 4, 0, 1, 2]))

    def _draw(self):
        x, ba = super()._draw()
        self.t += 1
        if self.t > self.drift_turn:
            ba = self.permutation[int(ba)]
        return x, ba


class MisleadingFeatureEnv(Contextual_Bandit_generator):
    """Break it: the logged context stops identifying the user.

    Two changes, both applied to the context handed to the agent while the
    reward still depends on the original hidden label:
      1. the one genuinely continuous informative feature (column 2) is
         overwritten with pure noise, destroying that signal outright;
      2. `n_noise` junk columns of unrelated noise are appended, so the real
         signal is buried among distractors.

    Why this is realistic: a logging change breaks one field so it writes
    garbage, and at the same time someone dumps a pile of unrelated columns
    into the feature store because storage is cheap. Nobody notices because
    the pipeline still runs.

    Assumption violated: that the context is informative about the reward. A
    contextual bandit is only better than a plain multi-armed bandit if X
    actually predicts which action pays. Strip that out and the agent is
    fitting forests to noise, which also costs it variance relative to just
    estimating a per-arm mean.
    """

    name = "misleading"

    def __init__(self, config, seed=None, n_noise=5, corrupt_cols=(2,)):
        super().__init__(config, seed=seed)
        self.n_noise = n_noise
        self.corrupt_cols = tuple(corrupt_cols)
        self.noise_rng = np.random.RandomState(0 if seed is None else seed + 991)

    def _draw(self):
        x, ba = super()._draw()
        x = np.asarray(x, dtype=float).copy()
        # Overwrite the informative continuous column with noise drawn from a
        # similar range, so the corruption is not obvious from the marginals.
        for c in self.corrupt_cols:
            x[c] = self.noise_rng.normal(0.0, 1.0)
        if self.n_noise > 0:
            junk = self.noise_rng.normal(0.0, 1.0, size=self.n_noise)
            x = np.concatenate([x, junk])
        return x, ba


class HiddenVariableEnv(Contextual_Bandit_generator):
    """Break it: an unobserved user segment decides the answer.

    With probability `p_segment` a user belongs to a latent segment B. For
    those users the best article is shifted by `shift` positions. The segment
    membership is never exposed in X, so from the agent's point of view two
    users with identical contexts want different articles.

    Why this is realistic: logged-out or cookie-less traffic, a device type
    that never made it into the feature pipeline, or intent the platform
    cannot observe. This is the single most common way real recommenders
    underperform their offline metrics.

    Assumption violated: that X is a sufficient statistic for the reward. The
    reward now depends on something outside the context, so P(click | x, a) is
    genuinely ambiguous and no function approximator on X can resolve it. The
    agent's best possible play becomes picking the article that the larger
    segment prefers, which caps performance well below the clean environment.
    """

    name = "hidden"

    def __init__(self, config, seed=None, p_segment=0.35, shift=2):
        super().__init__(config, seed=seed)
        self.p_segment = p_segment
        self.shift = shift
        self.seg_rng = np.random.RandomState(0 if seed is None else seed + 313)

    def _draw(self):
        x, ba = super()._draw()
        if self.seg_rng.random_sample() < self.p_segment:
            ba = (int(ba) + self.shift) % N_ACTIONS
        return x, ba


ENVIRONMENTS = {
    'baseline': Contextual_Bandit_generator,
    'nonstationary': NonStationaryEnv,
    'misleading': MisleadingFeatureEnv,
    'hidden': HiddenVariableEnv,
}


def make_env(env_name, n_samples=100000, seed=None, **kwargs):
    """Construct an environment by name. See ENVIRONMENTS."""
    return ENVIRONMENTS[env_name]({'n_samples': n_samples}, seed=seed, **kwargs)

## 6. Experiment harness

Runs every configuration and caches each finished run to `results/*.csv`
immediately. Re-running skips whatever is already on disk, so this is safe to
interrupt.

In [ ]:
SUMMARY_FIELDS = [
    'label', 'env', 'T', 'b', 'epsilon', 'seed', 'n_turns', 'window',
    'mean_reward', 'mean_reward_last', 'mean_reward_post_T',
    'match_rate', 'match_rate_last', 'n_refits', 'seconds',
]
CURVE_FIELDS = [
    'label', 'env', 'T', 'b', 'epsilon', 'seed', 'window', 'turn',
    'cum_avg_reward', 'rolling_avg_reward',
]

# Key identifying a run, used for resume. n_turns has to be part of this:
# without it, a short trial run would make the real sweep think it was already
# done and skip every configuration.
KEY = ('label', 'env', 'T', 'b', 'epsilon', 'seed', 'window', 'n_turns')

SUMMARY_CSV = 'sweep_summary.csv'
CURVE_CSV = 'sweep_curves.csv'


def cfg_of(label, env, T, b, epsilon, seed, n_turns, window=None,
           n_samples=100000):
    return {'label': label, 'env': env, 'T': T, 'b': b, 'epsilon': epsilon,
            'seed': seed, 'n_turns': n_turns, 'window': window,
            'n_samples': n_samples}


def make_key(cfg):
    return tuple(str(cfg.get(k, '') if cfg.get(k) is not None else '')
                 for k in KEY)


def append_rows(filename, fields, rows):
    """Append rows to a CSV, writing the header if the file is new."""
    if not rows:
        return
    path = os.path.join(RESULTS_DIR, filename)
    new = not os.path.exists(path) or os.path.getsize(path) == 0
    with open(path, 'a', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=fields)
        if new:
            w.writeheader()
        for row in rows:
            w.writerow(row)


def load_done_keys(filename):
    """Read back which runs are already on disk so they can be skipped."""
    path = os.path.join(RESULTS_DIR, filename)
    if not os.path.exists(path):
        return set()
    done = set()
    with open(path, newline='', encoding='utf-8') as fh:
        for row in csv.DictReader(fh):
            done.add(tuple(str(row.get(k, '')) for k in KEY))
    return done


def run_one(cfg):
    """Execute a single bandit run. Runs inside a worker process."""
    env = make_env(cfg['env'], n_samples=cfg['n_samples'], seed=cfg['seed'])
    out = run_bandit(env, T=cfg['T'], b=cfg['b'], epsilon=cfg['epsilon'],
                     n_turns=cfg['n_turns'], seed=cfg['seed'],
                     window=cfg['window'], rf_n_jobs=1)

    summary = {
        'label': cfg['label'], 'env': cfg['env'], 'T': cfg['T'],
        'b': cfg['b'], 'epsilon': cfg['epsilon'], 'seed': cfg['seed'],
        'n_turns': cfg['n_turns'],
        'window': cfg['window'] if cfg['window'] is not None else '',
        'mean_reward': round(out['mean_reward'], 6),
        'mean_reward_last': round(out['mean_reward_last'], 6),
        'mean_reward_post_T': round(float(out['mean_reward_post_T']), 6),
        'match_rate': round(out['match_rate'], 6),
        'match_rate_last': round(out['match_rate_last'], 6),
        'n_refits': out['n_refits'],
        'seconds': round(out['seconds'], 2),
    }

    # Two views of reward over time, downsampled so the curve file stays small.
    #   cum     : running average from turn 1, what the assignment asks for,
    #             but it smooths over anything that happens late
    #   rolling : trailing mean, the only way to actually see the
    #             non-stationary collapse and recovery
    rw = out['reward']
    n = len(rw)
    cum = np.cumsum(rw) / np.arange(1, n + 1)
    csum = np.concatenate([[0.0], np.cumsum(rw)])
    lo = np.maximum(0, np.arange(1, n + 1) - ROLLING_WINDOW)
    rolling = (csum[1:] - csum[lo]) / (np.arange(1, n + 1) - lo)

    idx = np.arange(CURVE_STRIDE - 1, n, CURVE_STRIDE)
    if len(idx) == 0 or idx[-1] != n - 1:
        idx = np.append(idx, n - 1)

    curves = [{
        'label': cfg['label'], 'env': cfg['env'], 'T': cfg['T'],
        'b': cfg['b'], 'epsilon': cfg['epsilon'], 'seed': cfg['seed'],
        'window': cfg['window'] if cfg['window'] is not None else '',
        'turn': int(i) + 1,
        'cum_avg_reward': round(float(cum[i]), 6),
        'rolling_avg_reward': round(float(rolling[i]), 6),
    } for i in idx]

    return summary, curves


def execute(configs, jobs):
    """Run configs in parallel, appending each result as it lands."""
    done = load_done_keys(SUMMARY_CSV)
    todo = [c for c in configs if make_key(c) not in done]

    if len(configs) - len(todo):
        print('  skipping %d run(s) already cached' % (len(configs) - len(todo)))
    if not todo:
        print('  nothing to do')
        return

    print('  %d run(s) to go on %d worker(s)' % (len(todo), jobs))
    started = time.perf_counter()
    finished = 0

    # max_nbytes=None disables joblib's array memmapping. The arrays here
    # are a couple of hundred KB, so memmapping buys nothing and its temp
    # folder cleanup is noisy on Windows.
    results = Parallel(n_jobs=jobs, backend='loky', max_nbytes=None,
                       return_as='generator_unordered')(
        delayed(run_one)(c) for c in todo)

    for summary, curves in results:
        append_rows(SUMMARY_CSV, SUMMARY_FIELDS, [summary])
        append_rows(CURVE_CSV, CURVE_FIELDS, curves)
        finished += 1
        elapsed = time.perf_counter() - started
        eta = (elapsed / finished) * (len(todo) - finished)
        print('  [%d/%d] %s T=%s b=%s eps=%s seed=%s w=%s -> mean=%.3f '
              'last1k=%.3f | %.1fm elapsed, eta %.1fm'
              % (finished, len(todo), summary['env'], summary['T'],
                 summary['b'], summary['epsilon'], summary['seed'],
                 summary['window'] or '-', summary['mean_reward'],
                 summary['mean_reward_last'], elapsed / 60, eta / 60),
              flush=True)


def build_all_configs(seeds=3, n_turns=N_TURNS):
    """Every run needed for Sections 7, 8 and 9."""
    configs = []

    # Section 7: the full 3 x 2 x 7 grid on the clean environment.
    for T, b, eps, seed in itertools.product(
            T_VALUES, B_VALUES, EPS_VALUES, range(seeds)):
        configs.append(cfg_of('sweep', 'baseline', T, b, eps, seed, n_turns))

    # Section 8: the same agent against each broken environment.
    for env_name in ['nonstationary', 'misleading', 'hidden']:
        for eps, seed in itertools.product(BROKEN_EPS, range(seeds)):
            configs.append(cfg_of('broken', env_name, BROKEN_T, BROKEN_B,
                                  eps, seed, n_turns))

    # Section 9: sliding-window fix vs the plain agent under drift.
    for eps, seed in itertools.product([0.05, 0.1, 0.2], range(seeds)):
        configs.append(cfg_of('fix_plain', 'nonstationary', BROKEN_T,
                              BROKEN_B, eps, seed, n_turns, window=None))
        for w in [500, 1000]:
            configs.append(cfg_of('fix_window', 'nonstationary', BROKEN_T,
                                  BROKEN_B, eps, seed, n_turns, window=w))
    return configs

### 6.1 Run the experiments

About 80 minutes the first time. After that the cached CSVs are reused, so
re-running this notebook is fast. Lower `JOBS` if the machine has fewer cores,
or set it to 1 if the parallel backend gives trouble.

In [ ]:
SEEDS = 3
JOBS = min(12, max(1, (os.cpu_count() or 4) - 2))

configs = build_all_configs(seeds=SEEDS)
print('total configurations:', len(configs))
execute(configs, JOBS)

sweep = pd.read_csv(os.path.join(RESULTS_DIR, SUMMARY_CSV))
curves = pd.read_csv(os.path.join(RESULTS_DIR, CURVE_CSV))
print('\nruns on disk:', len(sweep))

## 7. Sweep over T, b and epsilon

T in {100, 250, 500}, b in {100, 1000}, epsilon in {0, 0.05, 0.1, 0.2, 0.3,
0.4, 0.5}, 5000 turns, 3 seeds.

In [ ]:
base = sweep[(sweep['label'] == 'sweep') & (sweep['env'] == 'baseline')]

pivot = base.pivot_table(index=['T', 'b'], columns='epsilon',
                         values='mean_reward_last', aggfunc='mean')
print('Mean reward over the last 1000 turns:')
display(pivot.round(3))

In [ ]:
EPS_COLORS = {0.0: '#1f77b4', 0.05: '#ff7f0e', 0.1: '#2ca02c', 0.2: '#d62728',
              0.3: '#9467bd', 0.4: '#8c564b', 0.5: '#7f7f7f'}


def plot_sweep_curves(curves_df):
    """Average reward over time, faceted by T (rows) and b (columns)."""
    df = curves_df[(curves_df['label'] == 'sweep') &
                   (curves_df['env'] == 'baseline')]
    ts = sorted(df['T'].unique())
    bs = sorted(df['b'].unique())

    fig, axes = plt.subplots(len(ts), len(bs), figsize=(11, 3.2 * len(ts)),
                             sharex=True, sharey=True, squeeze=False)
    for i, T in enumerate(ts):
        for j, b in enumerate(bs):
            ax = axes[i][j]
            sub = df[(df['T'] == T) & (df['b'] == b)]
            for eps in sorted(sub['epsilon'].unique()):
                s = (sub[sub['epsilon'] == eps]
                     .groupby('turn')['cum_avg_reward'].mean())
                ax.plot(s.index, s.values, label='eps=%g' % eps,
                        color=EPS_COLORS.get(eps), linewidth=1.5)
            ax.axhline(RANDOM_POLICY_EXPECTED_REWARD, color='black',
                       linestyle='--', linewidth=1, alpha=0.6)
            ax.axvline(T, color='grey', linestyle=':', linewidth=1, alpha=0.7)
            ax.set_title('T=%d, b=%d' % (T, b), fontsize=10)
            ax.grid(alpha=0.25)
            if i == len(ts) - 1:
                ax.set_xlabel('turn')
            if j == 0:
                ax.set_ylabel('running average reward')

    axes[0][-1].legend(fontsize=8, loc='lower right', ncol=2)
    fig.suptitle('Learning curves (mean over seeds). Dashed = random policy '
                 '0.22, dotted = end of exploration', fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(os.path.join(FIG_DIR, 'q7_sweep_curves.png'), dpi=150,
                bbox_inches='tight')
    plt.show()


plot_sweep_curves(curves)

In [ ]:
def plot_sweep_final(summary_df):
    """Final performance across the whole grid, last 1000 turns."""
    df = summary_df[(summary_df['label'] == 'sweep') &
                    (summary_df['env'] == 'baseline')]
    ts = sorted(df['T'].unique())
    bs = sorted(df['b'].unique())
    epss = sorted(df['epsilon'].unique())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    width = 0.8 / len(ts)

    for j, b in enumerate(bs):
        ax = axes[j]
        x = np.arange(len(epss))
        for i, T in enumerate(ts):
            g = df[(df['T'] == T) & (df['b'] == b)].groupby('epsilon')[
                'mean_reward_last']
            means = [g.get_group(e).mean() if e in g.groups else np.nan
                     for e in epss]
            errs = [g.get_group(e).std(ddof=1)
                    if e in g.groups and len(g.get_group(e)) > 1 else 0
                    for e in epss]
            ax.bar(x + i * width - 0.4 + width / 2, means, width, yerr=errs,
                   capsize=3, label='T=%d' % T, edgecolor='black',
                   linewidth=0.4)
        ax.axhline(RANDOM_POLICY_EXPECTED_REWARD, color='crimson',
                   linestyle='--', linewidth=1.5, label='random policy 0.22')
        ax.set_xticks(x)
        ax.set_xticklabels(['%g' % e for e in epss])
        ax.set_xlabel('epsilon')
        ax.set_title('b = %d' % b)
        ax.grid(axis='y', alpha=0.3)

    axes[0].set_ylabel('mean reward, last 1000 turns')
    axes[0].legend(fontsize=9)
    fig.suptitle('Converged performance across T, b and epsilon', fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(os.path.join(FIG_DIR, 'q7_sweep_final_bars.png'), dpi=150,
                bbox_inches='tight')
    plt.show()


plot_sweep_final(sweep)

In [ ]:
agg = (base.groupby(['T', 'b', 'epsilon'])
           .agg(mean_all=('mean_reward', 'mean'),
                mean_last=('mean_reward_last', 'mean'),
                match_last=('match_rate_last', 'mean'))
           .reset_index()
           .sort_values('mean_last', ascending=False))
display(agg.round(3))

## 8. Effect of each environment modification

In [ ]:
broken = sweep[(sweep['T'] == BROKEN_T) & (sweep['b'] == BROKEN_B) &
               (sweep['label'].isin(['sweep', 'broken'])) &
               (sweep['window'].isna())]
tbl = broken.pivot_table(index='env', columns='epsilon',
                         values='mean_reward_last', aggfunc='mean')
order = [e for e in ['baseline', 'nonstationary', 'misleading', 'hidden']
         if e in tbl.index]
print('Mean reward over the last 1000 turns, by environment:')
display(tbl.loc[order].round(3))

print('\nDrop relative to the unmodified environment:')
display((tbl.loc[order] - tbl.loc['baseline']).round(3))

In [ ]:
ENV_LABELS = {
    'baseline': 'Baseline (unmodified)',
    'nonstationary': 'Non-stationary (preferences drift at t=2500)',
    'misleading': 'Misleading features (signal destroyed, junk added)',
    'hidden': 'Hidden variable (latent user segment)',
}
ENV_SHORT = {'baseline': 'Baseline', 'nonstationary': 'Non-stationary',
             'misleading': 'Misleading\nfeatures',
             'hidden': 'Hidden\nvariable'}


def plot_broken_comparison(summary_df, curves_df):
    """Baseline vs each broken environment at matched T, b and epsilon."""
    s = summary_df[(summary_df['T'] == BROKEN_T) &
                   (summary_df['b'] == BROKEN_B) &
                   (summary_df['label'].isin(['sweep', 'broken'])) &
                   (summary_df['window'].isna())]
    envs = [e for e in ['baseline', 'nonstationary', 'misleading', 'hidden']
            if e in set(s['env'])]
    epss = sorted(set(s[s['env'] == 'nonstationary']['epsilon'])) or \
        sorted(set(s['epsilon']))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

    ax = axes[0]
    x = np.arange(len(envs))
    width = 0.8 / len(epss)
    for i, eps in enumerate(epss):
        means, errs = [], []
        for env_name in envs:
            sub = s[(s['env'] == env_name) &
                    (s['epsilon'] == eps)]['mean_reward_last']
            means.append(sub.mean() if len(sub) else np.nan)
            errs.append(sub.std(ddof=1) if len(sub) > 1 else 0)
        ax.bar(x + i * width - 0.4 + width / 2, means, width, yerr=errs,
               capsize=3, label='eps=%g' % eps, color=EPS_COLORS.get(eps),
               edgecolor='black', linewidth=0.4)

    ax.axhline(RANDOM_POLICY_EXPECTED_REWARD, color='crimson', linestyle='--',
               linewidth=1.5, label='random policy 0.22')
    ax.set_xticks(x)
    ax.set_xticklabels([ENV_SHORT[e] for e in envs], fontsize=9)
    ax.set_ylabel('mean reward, last 1000 turns')
    ax.set_title('Effect of each environment modification\n(T=%d, b=%d)'
                 % (BROKEN_T, BROKEN_B), fontsize=10)
    ax.set_ylim(0, float(s['mean_reward_last'].max()) * 1.55)
    ax.legend(fontsize=8, ncol=3, loc='upper center', framealpha=0.95)
    ax.grid(axis='y', alpha=0.3)

    ax = axes[1]
    eps_ref = 0.1 if 0.1 in epss else epss[0]
    c = curves_df[(curves_df['T'] == BROKEN_T) &
                  (curves_df['b'] == BROKEN_B) &
                  (curves_df['epsilon'] == eps_ref) &
                  (curves_df['window'].isna()) &
                  (curves_df['label'].isin(['sweep', 'broken']))]
    # Rolling mean so the non-stationary collapse is actually visible.
    for env_name in envs:
        sub = c[c['env'] == env_name].groupby('turn')[
            'rolling_avg_reward'].mean()
        if len(sub):
            ax.plot(sub.index, sub.values, linewidth=1.8,
                    label=ENV_LABELS[env_name])
    ax.axhline(RANDOM_POLICY_EXPECTED_REWARD, color='black', linestyle='--',
               linewidth=1, alpha=0.6, label='random policy 0.22')
    ax.axvline(DRIFT_TURN, color='grey', linestyle=':', linewidth=1.5)
    ax.set_xlabel('turn')
    ax.set_ylabel('trailing %d-turn average reward' % ROLLING_WINDOW)
    ax.set_title('Learning curves at epsilon = %g' % eps_ref, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'q9_broken_environments.png'), dpi=150,
                bbox_inches='tight')
    plt.show()


plot_broken_comparison(sweep, curves)

## 9. Improving the agent for non-stationarity

Refit on a sliding window of the most recent W observations instead of the
whole history, via the `window` argument to `run_bandit`.

In [ ]:
fix = sweep[sweep['label'].isin(['fix_plain', 'fix_window'])].copy()
fix['variant'] = fix['window'].apply(
    lambda w: 'no window' if pd.isna(w) else 'window=%d' % w)
tbl = fix.pivot_table(index='variant', columns='epsilon',
                      values='mean_reward_last', aggfunc='mean')
print('Post-drift performance (mean reward over the last 1000 turns):')
display(tbl.round(3))

In [ ]:
def plot_fix(summary_df, curves_df):
    """Sliding-window agent vs the plain agent under drift."""
    s = summary_df[summary_df['label'].isin(['fix_plain', 'fix_window'])]
    epss = sorted(s['epsilon'].unique())
    eps_ref = 0.1 if 0.1 in epss else epss[0]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

    ax = axes[0]
    c = curves_df[(curves_df['label'].isin(['fix_plain', 'fix_window'])) &
                  (curves_df['epsilon'] == eps_ref)]
    # Rolling, not cumulative: a cumulative average from turn 1 badly
    # understates the collapse, because 2500 turns of good pre-drift history
    # keep propping the number up.
    for w, grp in c.groupby(c['window'].fillna(-1)):
        name = 'no window (all history)' if w == -1 else 'window = %d' % int(w)
        m = grp.groupby('turn')['rolling_avg_reward'].mean()
        ax.plot(m.index, m.values, linewidth=1.8, label=name)

    ax.axvline(DRIFT_TURN, color='grey', linestyle=':', linewidth=1.5)
    ax.axhline(RANDOM_POLICY_EXPECTED_REWARD, color='black', linestyle='--',
               linewidth=1, alpha=0.6, label='random policy 0.22')
    ax.set_xlabel('turn')
    ax.set_ylabel('trailing %d-turn average reward' % ROLLING_WINDOW)
    ax.set_title('Recovery after the drift, epsilon = %g\n'
                 '(dotted line marks the drift at t=%d)'
                 % (eps_ref, DRIFT_TURN), fontsize=10)
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(alpha=0.25)

    ax = axes[1]
    x = np.arange(len(epss))
    variants = sorted(s['window'].fillna(-1).unique())
    width = 0.8 / len(variants)
    for i, w in enumerate(variants):
        name = 'no window' if w == -1 else 'window = %d' % int(w)
        means, errs = [], []
        for eps in epss:
            sub = s[(s['window'].fillna(-1) == w) &
                    (s['epsilon'] == eps)]['mean_reward_last']
            means.append(sub.mean() if len(sub) else np.nan)
            errs.append(sub.std(ddof=1) if len(sub) > 1 else 0)
        ax.bar(x + i * width - 0.4 + width / 2, means, width, yerr=errs,
               capsize=3, label=name, edgecolor='black', linewidth=0.4)

    ax.axhline(RANDOM_POLICY_EXPECTED_REWARD, color='crimson', linestyle='--',
               linewidth=1.5, label='random policy 0.22')
    ax.set_xticks(x)
    ax.set_xticklabels(['%g' % e for e in epss])
    ax.set_xlabel('epsilon')
    ax.set_ylabel('mean reward, last 1000 turns')
    ax.set_title('Post-drift performance (last 1000 turns)', fontsize=10)
    ax.set_ylim(0, float(s['mean_reward_last'].max()) * 1.45)
    ax.legend(fontsize=8, loc='upper center', ncol=4, framealpha=0.95)
    ax.grid(axis='y', alpha=0.3)

    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'q9_nonstationary_fix.png'), dpi=150,
                bbox_inches='tight')
    plt.show()


plot_fix(sweep, curves)